In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt

from propofol.haemo_pd_su2023 import SuHaemoPD
from propofol.patient import EleveldPatient as Patient
from propofol.propofol_pkpd import EleveldPK as PropofolPK
from propofol.propofol_pkpd import EleveldPD as PropofolPD
from propofol.remifentanil_pkpd import EleveldPK as RemifentanilPK


# ============================================================
# 1. Patient
# ============================================================
patient_35 = Patient(
    age=35,
    height=170,
    weight=70,
    sex="male",
    opiates=False,
    base_sv=82.2,
    base_hr=56.0,
    base_tpr=0.016,
)


# ============================================================
# 2. Base models
# ============================================================
pk_propofol = PropofolPK(patient_35)
pd_propofol = PropofolPD(patient_35)
pk_remifentanil = RemifentanilPK(patient_35)

model_full = SuHaemoPD(
    patient=patient_35,
    pk_propofol=pk_propofol,
    pd_propofol=pd_propofol,
    pk_remifentanil=pk_remifentanil,
    use_bsv=False,
)


# ============================================================
# 3. Time grid for convergence
# ============================================================
t = np.linspace(0.0, 180.0, 180 * 60 + 1)  # long enough to converge


# ============================================================
# 4. Helper: freeze PK so Cp stays constant
# ============================================================
def make_constant_concentration_model(model):
    """
    Return a deepcopy with PK transfer/elimination set to zero so that
    A1 and A4 remain fixed and therefore Cp stays constant.
    """
    m = copy.deepcopy(model)

    # Propofol PK frozen
    m.pk_propofol.k10 = 0.0
    m.pk_propofol.k12 = 0.0
    m.pk_propofol.k13 = 0.0
    m.pk_propofol.k21 = 0.0
    m.pk_propofol.k31 = 0.0

    # Propofol effect-site also frozen so Ce stays at initial value
    if hasattr(m.pd_propofol, "ke0"):
        m.pd_propofol.ke0 = 0.0

    # Remifentanil PK frozen
    m.pk_remifentanil.k10 = 0.0
    m.pk_remifentanil.k12 = 0.0
    m.pk_remifentanil.k13 = 0.0
    m.pk_remifentanil.k21 = 0.0
    m.pk_remifentanil.k31 = 0.0

    return m


# ============================================================
# 5. Helper: simulate long enough for haemodynamics to converge
# ============================================================
def simulate_constant_cp(model, cp_prop, cp_remi, t):
    """
    Hold Cp_prop and Cp_remi constant by fixing A1 and A4 and freezing PK.
    Then integrate haemodynamic ODEs to steady state.
    """
    m = make_constant_concentration_model(model)

    # NONMEM-consistent initial conditions
    tde_sv0 = m.base_sv * m.ltde_sv
    tde_hr0 = m.base_hr * m.ltde_hr

    y0 = [
        cp_prop * m.pk_propofol.V1,   # A1 fixed => constant propofol Cp
        0.0,                          # A2
        0.0,                          # A3
        0.0,                          # Ce_prop
        cp_remi * m.pk_remifentanil.V1,  # A4 fixed => constant remi Cp
        0.0,                          # A5
        0.0,                          # A6
        m.base_sv,                    # sv_ast
        m.base_hr,                    # hr_ast
        m.base_tpr,                   # tpr
        tde_sv0,                      # tde_sv
        tde_hr0,                      # tde_hr
    ]

    (
        A1, A2, A3, Ce_prop,
        A4, A5, A6,
        sv_ast, hr_ast, tpr,
        tde_sv, tde_hr,
        sv, MAP,
    ) = m.solve_ode(
        t=t,
        y0=y0,
        dosing_prop=None,
        dosing_remi=None,
    )

    hr = hr_ast + tde_hr

    return {
        "cp_prop": float(cp_prop),
        "cp_remi": float(cp_remi),
        "sv_ast": float(sv_ast[-1]),
        "hr_ast": float(hr_ast[-1]),
        "tpr": float(tpr[-1]),
        "tde_sv": float(tde_sv[-1]),
        "tde_hr": float(tde_hr[-1]),
        "sv": float(sv[-1]),
        "hr": float(hr[-1]),
        "map": float(MAP[-1]),
    }


def solve_curve_by_ode(model, cp_prop_grid, cp_remi, t):
    """
    Build a concentration-effect curve by long ODE simulation at each fixed Cp.
    """
    map_vals = []
    hr_vals = []
    sv_vals = []
    sv_ast_vals = []
    hr_ast_vals = []
    tpr_vals = []

    for cp in cp_prop_grid:
        out = simulate_constant_cp(
            model=model,
            cp_prop=cp,
            cp_remi=cp_remi,
            t=t,
        )
        map_vals.append(out["map"])
        hr_vals.append(out["hr"])
        sv_vals.append(out["sv"])
        sv_ast_vals.append(out["sv_ast"])
        hr_ast_vals.append(out["hr_ast"])
        tpr_vals.append(out["tpr"])

    return {
        "map": np.asarray(map_vals, dtype=float),
        "hr": np.asarray(hr_vals, dtype=float),
        "sv": np.asarray(sv_vals, dtype=float),
        "sv_ast": np.asarray(sv_ast_vals, dtype=float),
        "hr_ast": np.asarray(hr_ast_vals, dtype=float),
        "tpr": np.asarray(tpr_vals, dtype=float),
    }


def pct_change(y, baseline):
    y = np.asarray(y, dtype=float)
    return 100.0 * (y - baseline) / baseline


def additive_null_curve(prop_curve, remi_only, baseline_prop_only):
    """
    Additive null interaction in output space.
    """
    return {
        "map": prop_curve["map"] + remi_only["map"] - baseline_prop_only["map"],
        "hr": prop_curve["hr"] + remi_only["hr"] - baseline_prop_only["hr"],
        "sv": prop_curve["sv"] + remi_only["sv"] - baseline_prop_only["sv"],
    }


# ============================================================
# 6. Concentration grid
# ============================================================
cp_prop_grid = np.linspace(0.0, 10.0, 300)
cp_remi_fixed = 2.0


# ============================================================
# 7. Baselines from long ODE simulation
# ============================================================
baseline_prop_only = simulate_constant_cp(
    model=model_full,
    cp_prop=0.0,
    cp_remi=0.0,
    t=t,
)

baseline_full = simulate_constant_cp(
    model=model_full,
    cp_prop=0.0,
    cp_remi=cp_remi_fixed,
    t=t,
)

remi_only = simulate_constant_cp(
    model=model_full,
    cp_prop=0.0,
    cp_remi=cp_remi_fixed,
    t=t,
)


# ============================================================
# 8. Curves from long ODE simulation
# ============================================================
curve_prop_only = solve_curve_by_ode(
    model=model_full,
    cp_prop_grid=cp_prop_grid,
    cp_remi=0.0,
    t=t,
)

curve_full = solve_curve_by_ode(
    model=model_full,
    cp_prop_grid=cp_prop_grid,
    cp_remi=cp_remi_fixed,
    t=t,
)

curve_null = additive_null_curve(
    prop_curve=curve_prop_only,
    remi_only=remi_only,
    baseline_prop_only=baseline_prop_only,
)

baseline_null = {
    "map": float(curve_null["map"][0]),
    "hr": float(curve_null["hr"][0]),
    "sv": float(curve_null["sv"][0]),
}


# ============================================================
# 9. Percentage change from scenario-specific baselines
# ============================================================
map_prop_only = pct_change(curve_prop_only["map"], baseline_prop_only["map"])
hr_prop_only = pct_change(curve_prop_only["hr"], baseline_prop_only["hr"])
sv_prop_only = pct_change(curve_prop_only["sv"], baseline_prop_only["sv"])

map_null = pct_change(curve_null["map"], baseline_null["map"])
hr_null = pct_change(curve_null["hr"], baseline_null["hr"])
sv_null = pct_change(curve_null["sv"], baseline_null["sv"])

map_full = pct_change(curve_full["map"], baseline_full["map"])
hr_full = pct_change(curve_full["hr"], baseline_full["hr"])
sv_full = pct_change(curve_full["sv"], baseline_full["sv"])

# ============================================================
# 12. Plot
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharex=True)

purple = "purple"
green = "green"
blue = "blue"

axes[0].plot(cp_prop_grid, map_prop_only, color=purple, lw=2, label="Propofol alone")
axes[0].plot(cp_prop_grid, map_null, color=green, lw=2, label="Combination - null interaction")
axes[0].plot(cp_prop_grid, map_full, color=blue, lw=2, label="Combination interaction")
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_xlabel("Propofol plasma concentration (µg/mL)")
axes[0].set_ylabel("MAP change from baseline (%)")

axes[1].plot(cp_prop_grid, hr_prop_only, color=purple, lw=2)
axes[1].plot(cp_prop_grid, hr_null, color=green, lw=2)
axes[1].plot(cp_prop_grid, hr_full, color=blue, lw=2)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_xlabel("Propofol plasma concentration (µg/mL)")
axes[1].set_ylabel("HR change from baseline (%)")

axes[2].plot(cp_prop_grid, sv_prop_only, color=purple, lw=2)
axes[2].plot(cp_prop_grid, sv_null, color=green, lw=2)
axes[2].plot(cp_prop_grid, sv_full, color=blue, lw=2)
axes[2].axhline(0, color="black", lw=0.8)
axes[2].set_xlabel("Propofol plasma concentration (µg/mL)")
axes[2].set_ylabel("SV change from baseline (%)")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, -0.03),
)
fig.tight_layout()
plt.show()